<a href="https://colab.research.google.com/github/jonathanahc/Ejercicios/blob/main/Relaciones_entre_Clases_y_Herencia_M%C3%BAltiple_en_C%2B%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Relaciones entre clases y herencia múltiple en C++**


## **1 · El problema inicial: clases aisladas**

Imaginemos que definimos `Alumno`, `Materia`, `Profesor`, `Laboratorio` y `Equipo`. ¿Basta con declarar las clases para describir la universidad? **No:** falta representar quién se relaciona con quién.

La pregunta que nos guiará es: **¿el objeto solo interactúa con otro, lo agrupa, es su dueño o es un tipo de otro?**

| Relación | Pregunta guía | Ejemplo del sistema |
|---|---|---|
| **Asociación** | ¿Trabajan juntos? | Alumno cursa Materia |
| **Agregación** | ¿El grupo reúne objetos independientes? | Laboratorio reúne Profesores |
| **Composición** | ¿El objeto posee y administra sus componentes? | Equipo tiene MóduloInterno |
| **Herencia simple** | ¿Es un tipo particular de otro? | Investigador es Persona |
| **Herencia múltiple** | ¿Combina dos tipos o capacidades? | EquipoMultifuncional hereda Imprimible y Escaneable |

**Punto de partida:** empezamos por la relación menos comprometida: dos objetos que solamente colaboran.

## **2 · Asociación: «trabaja con»**

Un `Alumno` cursa una `Materia`. Ambos pueden existir por separado. La materia **no pertenece** al alumno y el alumno **no pertenece** a la materia.

**UML:** `Alumno ───── Materia` (línea simple).

En C++, una forma de mostrar esta colaboración es pasar un objeto por **referencia** (`Materia&`) a un método. La relación en este ejemplo ocurre durante la llamada; no queda almacenada permanentemente en el alumno.

In [1]:
%%writefile asociacion.cpp
#include <iostream>
#include <string>
using namespace std;

class Materia
{
public:
    string nombre;
    explicit Materia(string n) : nombre(n) {}
};

class Alumno
{
public:
    string nombre;
    explicit Alumno(string n) : nombre(n) {}
    void cursar(const Materia& materia) const {
        cout << nombre << " cursa " << materia.nombre << "\n";
    }
};

int main()
{
    Alumno alumno("Ana");
    Materia poo("POO");
    alumno.cursar(poo);
    cout << "La materia sigue existiendo: " << poo.nombre << "\n";
}

Writing asociacion.cpp


In [2]:
!g++ -std=c++17 -Wall -Wextra asociacion.cpp -o asociacion && ./asociacion

Ana cursa POO
La materia sigue existiendo: POO


Ahora el centro de investigación quiere **agrupar** a sus profesores dentro de un laboratorio. Ya no basta con colaborar durante una llamada: el laboratorio debe guardar una colección de profesores. Pero los profesores deben poder cambiar de laboratorio sin ser destruidos. Esto nos lleva a la **agregación**.

**¿Qué desaparece si termina la ejecución del método `cursar()`? ¿Se destruye `poo`?**

## **3 · Agregación: «agrupa, pero no posee»**

Un `Laboratorio` reúne profesores contratados por la universidad. Los profesores existen **independientemente** del laboratorio. Si el laboratorio se cierra, los profesores no son destruidos por él.

**UML:** `Laboratorio ◇──── Profesor` (rombo **blanco** en el lado del todo).

En el ejemplo, el laboratorio almacena **punteros no propietarios** a profesores creados en otro lugar. Los profesores deben seguir vivos mientras el laboratorio utilice esos punteros.

In [3]:
%%writefile agregacion.cpp
#include <iostream>
#include <string>
#include <vector>
using namespace std;

class Profesor
{

public:
    string nombre;
    explicit Profesor(string n) : nombre(n) {}
};

class Laboratorio
{
    vector<Profesor*> profesores; // No es dueño de los profesores
public:
    void incorporar(Profesor& p) { profesores.push_back(&p); }
    void mostrar() const
    {
        for (const Profesor* p : profesores)
            cout << "Profesor: " << p->nombre << "\n";
    }
};

int main()
{
    Profesor ana("Ana"), luis("Luis");
    {
        Laboratorio ia;
        ia.incorporar(ana);
        ia.incorporar(luis);
        ia.mostrar();
    } // El laboratorio deja de existir
    cout << "Profesor disponible después: " << ana.nombre << "\n";
}

Writing agregacion.cpp


In [4]:
!g++ -std=c++17 -Wall -Wextra agregacion.cpp -o agregacion && ./agregacion

Profesor: Ana
Profesor: Luis
Profesor disponible después: Ana


**¿Qué cambió respecto de asociación?** Ahora `Laboratorio` **mantiene referencias** a objetos ajenos para formar un grupo. En ambos casos los participantes son independientes, pero la agregación representa una relación **todo–parte débil**.

**¿Y si el laboratorio incorpora un equipo cuya tarjeta de control se fabrica y se destruye con él? En ese caso sí queremos **propiedad y ciclo de vida compartido**. Aparece la composición.**

## **4 · Composición: «contiene y es responsable de»**

Un `Equipo` contiene un `ModuloInterno`. No se trata solo de almacenar una dirección externa: el módulo se construye **como parte del equipo** y se destruye con él.

**UML:** `Equipo ◆──── ModuloInterno` (rombo **negro** del lado del todo).

> En diseño, composición significa **propiedad exclusiva del componente en ese modelo**. Una pieza física podría sobrevivir a su máquina, pero aquí modelamos un módulo interno perteneciente al objeto `Equipo`.

**Observa el orden de construcción y destrucción en C++.**

In [5]:
%%writefile composicion.cpp
#include <iostream>
using namespace std;

class ModuloInterno
{
public:
    ModuloInterno() { cout << "1. Se construye el modulo\n"; }
    ~ModuloInterno() { cout << "4. Se destruye el modulo\n"; }
};

class Equipo
{
    ModuloInterno modulo; // Miembro por valor: propiedad y duración ligada al Equipo
public:
    Equipo() { cout << "2. Se construye el equipo\n"; }
    ~Equipo() { cout << "3. Se destruye el equipo\n"; }
};

int main()
{
    { Equipo microscopio; }
    cout << "El bloque termino.\n";
}

Writing composicion.cpp


In [6]:
!g++ -std=c++17 -Wall -Wextra composicion.cpp -o composicion && ./composicion

1. Se construye el modulo
2. Se construye el equipo
3. Se destruye el equipo
4. Se destruye el modulo
El bloque termino.


### **Alto y compara**

| Caso | ¿Relación todo–parte? | ¿Quién controla la vida del otro? | Decisión |
|---|---|---|---|
| Alumno usa Materia | No necesariamente | Ninguno | **Asociación** |
| Laboratorio reúne Profesores | Sí | El laboratorio **no** destruye profesores | **Agregación** |
| Equipo integra MóduloInterno | Sí | El equipo controla el módulo | **Composición** |

**Importante:** ninguna palabra por sí sola («tiene», «usa», «contiene») determina automáticamente la relación. Debes conocer las reglas del sistema: **propiedad, independencia y ciclo de vida**.

**Hasta ahora describimos **cómo un objeto se vincula con otro**. Pero ¿cómo indicamos que `Investigador` **es un tipo de** `Persona`? Esa no es una relación todo–parte: es **herencia**.**

## **5 · Herencia simple: «es un»**

Una persona puede tener nombre; un investigador, además, puede investigar. **No decimos que el investigador contiene una persona**, sino que **es una persona**.

**UML:** `Investigador ────▷ Persona` (triángulo **vacío apuntando a la base**).

En C++, `class Investigador : public Persona` reutiliza las operaciones accesibles de `Persona`. Esto es **herencia simple**: una clase derivada, una clase base.

In [7]:
%%writefile herencia_simple.cpp
#include <iostream>
#include <string>
using namespace std;

class Persona
{
protected:
    string nombre;
public:
    explicit Persona(string n) : nombre(n) {}
    void presentarse() const { cout << "Soy " << nombre << "\n"; }
};

class Investigador : public Persona
{
public:
    explicit Investigador(string n) : Persona(n) {}
    void investigar() const { cout << nombre << " realiza investigacion\n"; }
};

int main()
{
    Investigador ana("Ana");
    ana.presentarse(); // Heredado
    ana.investigar();  // Propio
}

Writing herencia_simple.cpp


In [8]:
!g++ -std=c++17 -Wall -Wextra herencia_simple.cpp -o herencia_simple && ./herencia_simple

Soy Ana
Ana realiza investigacion


**En el centro llega una estación que **puede imprimir** resultados y **puede escanear** muestras. Si ambas capacidades ya están definidas como clases base independientes, C++ permite derivar de **las dos**. Esa es la herencia múltiple.**

## **6 · Herencia múltiple: dos bases, una derivada**

Una estación multifuncional combina dos capacidades:

```text
Imprimible      Escaneable
    △               △
     \             /
      \           /
     EstacionMultifuncional
```

La declaración separa las clases base por comas:

`class EstacionMultifuncional : public Imprimible, public Escaneable { ... };`

Observa que la estación puede ejecutar métodos de **ambas bases** y definir los suyos.

In [9]:
%%writefile herencia_multiple.cpp
#include <iostream>
using namespace std;

class Imprimible
{
public:
    void imprimir() const { cout << "Imprimiendo resultados\n"; }
};

class Escaneable
{
public:
    void escanear() const { cout << "Escaneando muestras\n"; }
};

class EstacionMultifuncional : public Imprimible, public Escaneable
{
public:
    void prepararInforme() const {
        escanear();
        imprimir();
        cout << "Informe preparado\n";
    }
};

int main()
{
    EstacionMultifuncional estacion;
    estacion.imprimir();
    estacion.escanear();
    estacion.prepararInforme();
}

Writing herencia_multiple.cpp


In [10]:
!g++ -std=c++17 -Wall -Wextra herencia_multiple.cpp -o herencia_multiple && ./herencia_multiple

Imprimiendo resultados
Escaneando muestras
Escaneando muestras
Imprimiendo resultados
Informe preparado


## **7 · Una consecuencia: nombres ambiguos**

La reutilización tiene un costo: ¿qué ocurre si **las dos bases** declaran un método llamado `estado()`?

La llamada `objeto.estado()` sería **ambigua**. Para evitar que falle el notebook, primero mostramos **la llamada problemática comentada**, y después la solución mediante el operador de resolución de ámbito `::`.

No es que el compilador elija al azar: exige que indiquemos a qué método nos referimos.

In [ ]:
%%writefile ambiguedad.cpp
#include <iostream>
using namespace std;

class Imprimible
{
public:
    void estado() const { cout << "Estado de impresora\n"; }
};

class Escaneable
{
public:
    void estado() const { cout << "Estado de escaner\n"; }
};

class Estacion : public Imprimible, public Escaneable {};

int main()
{
    Estacion estacion;
    // estacion.estado(); // Error: llamada ambigua
    estacion.Imprimible::estado();
    estacion.Escaneable::estado();
}

In [ ]:
!g++ -std=c++17 -Wall -Wextra ambiguedad.cpp -o ambiguedad && ./ambiguedad